In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
%run /Workspace/fmcg_domain/Setup/utilities

In [0]:
bronze_schema,silver_schema,gold_schema
dbutils.widgets.text("catalog","fmcg","Catalog")
dbutils.widgets.text("datasource","customers","Data source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("datasource")

In [0]:
silver_customer_table = f"{catalog}.{silver_schema}.customers"
gold_customer_table = f"{catalog}.{gold_schema}.{data_source}"

gold_customer_table



In [0]:
df = spark.sql(f"select * from {silver_customer_table}")
display(df,10)

In [0]:
gold_df = df.select("customer_code","customer","market","platform","channel","city")

display(gold_df)

In [0]:
gold_df.write.format("delta")\
    .option("delta.enableChangeDataFeed","true")\
    .option("mergeSchema","true")\
    .saveAsTable(gold_customer_table)
display(gold_df)

In [0]:
delta_table = DeltaTable.forName(spark,"fmcg.gold.dim_customers")

child_table = DeltaTable.forName(spark,gold_customer_table)

In [0]:
delta_table.alias("target").merge(
    child_table.toDF().alias("source"),
    "target.customer_code = source.customer_code"
).whenMatchedUpdateAll()\
.whenNotMatchedInsertAll().execute()

In [0]:
df = spark.sql("select * from fmcg.gold.dim_customers")
display(df)